# Sadeghi et al. (2019): auditory model claims (CRCNS bst-2)

Paper: Sadeghi M, Zhai X, Stevenson IH, Escabi MA (2019). A neural ensemble correlation code for
sound category identification. PLoS Biol 17(10):e3000449, doi:10.1371/journal.pbio.3000449. The
claims file (claims/sadeghi-2019.yaml) was written from the version of record (PMC6788721) with
its supplement.

Dataset: CRCNS bst-2 (doi:10.6080/K03X84V3), bst-2/live/data.tar.gz in the public bucket
crcnsarchive, `crcns:bst-2/unversioned` (CRCNS does not version data sets; the paper cites the DOI only, and
this is the bucket's only copy). MD5 of data.tar.gz from bst-2/live/checksums.md5:
916ae724cd3334a16a4396b861e4abe0. The model sounds are data/Sounds/ModelSounds/<category>/Sound_<n>.wav.

This notebook tests the auditory model claims; the neural claims are in
reproduce_sadeghi_2019_neural.ipynb.

| Claim | Tested here |
|---|---|
| fig5-speech-water-widths | tested |
| fig6-category-half-widths | tested |
| fig6c-category-diversity | tested |
| fig7i-stationarity | tested |
| fig9a-optimal-resolution | blocked: too large for this run (estimate below) |
| fig9b-similar-maximum | blocked: too large (same classifier) |
| fig9b-accumulation-rate | blocked: too large (same classifier) |
| fig9c-time-averaged-worse | blocked: too large (same classifier) |
| fig9e-category-specific | blocked: too large (same classifier) |
| fig9-spectrum-classifier-lower | blocked: too large (same classifier) |
| fig10-vocalization-background | blocked: too large (same classifier) |

Method: the cochlear model and short-term correlations follow the Methods ("Auditory model",
"Nonstationary spectro-temporal correlation statistics", "Stationarity and ensemble diversity
indices") and port the paper's code in the data set (Code/AuditoryModelCorr); model.py says
where each parameter comes from. White noise is generated, as the data set does not hold it.

In [ ]:
import time, resource
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import sadeghi as S
import model as M

T_START = time.time()
SEED_NOISE = 505  # standard choice, white noise control

## Loading

The 195 model sounds are the first members of the tar stream after the paradigm sounds; the
stream is closed once they are read. White noise, which the data set does not hold, is
generated (15 Gaussian noises of 10 s, as in the claims file).

In [ ]:
raw = {}
for name, b in S.stream_members(lambda n: "/Sounds/ModelSounds/" in n and n.endswith(".wav")):
    raw[name] = b
    if len(raw) == 195:
        break
CATS = sorted({n.split("/")[-2] for n in raw})
sounds = {}
for n, b in raw.items():
    fs, x = M.read_wav(b)
    sounds[(n.split("/")[-2], int(n.split("_")[-1][:-4]))] = (fs, x, b[:44])
rng = np.random.default_rng(SEED_NOISE)
for i in range(1, 16):
    sounds[("White_Noise", i)] = (44100, rng.standard_normal(441000) * 0.001, b"")
print("read", len(raw), "files in", round(time.time() - T_START), "s")

In [ ]:
print("categories:", len(CATS), CATS)
for c in CATS:
    keys = sorted(k for k in sounds if k[0] == c)
    fss = {sounds[k][0] for k in keys}
    lens = {len(sounds[k][1]) / sounds[k][0] for k in keys}
    sds = [sounds[k][1].std() for k in keys]
    print(f"{c:20s} sounds {len(keys):2d} (numbers {keys[0][1]}-{keys[-1][1]}) fs {fss} duration (s) {sorted(lens)}"
          f" SD (int16 units) {min(sds):.1f}-{max(sds):.1f}")
hdr = sounds[(CATS[0], 1)][2]
print("WAV header of one file: format tag", int.from_bytes(hdr[20:22], "little"), "bits", int.from_bytes(hdr[34:36], "little"),
      "(1 = integer PCM; the data description says double precision, normalised to SD 0.001)")
print("SD as a fraction of full scale, all model sounds:",
      round(min(sounds[k][1].std() for k in sounds if k[0] != "White_Noise") / 32768, 5), "to",
      round(max(sounds[k][1].std() for k in sounds if k[0] != "White_Noise") / 32768, 5))

The files are 16 bit integer PCM, not double precision as the data description says; their SD
as a fraction of full scale is about 0.001, as it says. Every sound is 10 s at 44.1 kHz and
every category has 15, as the Methods state.

## Model correlations of every sound

Per sound: the cochlear envelopes (58 channels, 1 kHz); the short-term spectro-temporal
correlation at tau_W = 100 ms with 99 lags (for SI, CDI and the spectral breadth); the
time-averaged short-term autocorrelations at tau_W = 566 ms (lags within 283 ms); and the
long-term autocorrelation of the whole envelopes (lags within 283 ms).

In [ ]:
LAG566 = np.arange(-283, 284).astype(float)
cbar, si_terms, auto566, autolong, zero_long = {}, {}, {}, {}, {}
t0 = time.time()
for i, key in enumerate(sorted(sounds)):
    fs, x, _ = sounds[key]
    Sg = M.cochleogram(x, fs)
    n, s1, s2 = 0, 0.0, 0.0
    acc = None
    for c in M.short_term_iter(Sg, 100, 49, pairs="all"):
        v = c.astype(np.float64)
        acc = v.copy() if acc is None else acc + v
        s2 += (v ** 2).sum()
        n += 1
    cbar[key] = (acc / n).astype(np.float32)
    si_terms[key] = (s2 / n, float((cbar[key].astype(np.float64) ** 2).sum()), n)
    auto566[key] = np.mean(list(M.short_term_iter(Sg, 566, 283, pairs="auto")), 0)
    autolong[key] = M.long_term_auto(Sg, 283)
    X = Sg - Sg.mean(1, keepdims=True)
    zero_long[key] = np.corrcoef(X)
    if i % 30 == 0:
        print(i, key, f"{time.time() - t0:.0f} s", flush=True)
print("time points per sound at 100 ms:", sorted({v[2] for v in si_terms.values()}))
print(f"done in {(time.time() - t0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## fig5-speech-water-widths

> In the cochlear model, the example speech sound has broad spectral correlations and a slow
> temporal correlation (64 ms half width), while the example water sound has diagonal
> spectral correlations and a fast temporal correlation (4 ms half width).

Readings: window (ii)(a) time average of the short-term correlation at tau_W = 566 ms,
(ii)(b) long-term correlation of the whole envelopes; width (iii)(a) full width at 50% of the
channel-averaged autocorrelation, (iii)(b) median over channels. Spectral breadth is the mean
off-diagonal zero-lag correlation (time-averaged at 100 ms, and long-term).

In [ ]:
def widths(curves):
    return M.width_at(curves.mean(0), LAG566, 0.5), float(np.median([M.width_at(c, LAG566, 0.5) for c in curves]))

def breadth(key):
    off = ~np.eye(58, dtype=bool)
    return float(cbar[key][:, :, 49][off].mean()), float(zero_long[key][off].mean())

SP, WA = ("Human_Speech", 1), ("Flowing_Water", 1)
for lab, src in (("(ii)(a) 566 ms short-term", auto566), ("(ii)(b) long-term", autolong)):
    ws, wa = widths(src[SP]), widths(src[WA])
    for j, wl in enumerate(("(iii)(a) mean curve", "(iii)(b) median of channels")):
        print(f"{lab}, {wl}: speech {ws[j]:.1f} ms (64 +/- 6.4: {abs(ws[j] - 64) <= 6.4}), water {wa[j]:.1f} ms (4 +/- 1: {abs(wa[j] - 4) <= 1})")
print("spectral breadth (100 ms time average, long-term): speech", np.round(breadth(SP), 3), "water", np.round(breadth(WA), 3))
print("variant, every sound of the category, (ii)(a)(iii)(a):")
for cat in ("Human_Speech", "Flowing_Water"):
    v = [widths(auto566[(cat, i)])[0] for i in range(1, 16)]
    print(f"   {cat}: {np.round(v, 1)}")
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
for a, key in zip(ax, (SP, WA)):
    a.plot(LAG566, auto566[key].mean(0), label="566 ms short-term"); a.plot(LAG566, autolong[key].mean(0), label="long-term")
    a.set_title(key[0] + " 1"); a.set_xlabel("lag (ms)")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
print("plotted range:", round(float(min(auto566[SP].mean(0).min(), autolong[WA].mean(0).min())), 3), "to 1")

**Result: with the time-averaged short-term autocorrelation at tau_W = 566 ms, the speech example's half width is 64.5 ms and the water example's 4.0 ms (median over channels 70.0 and 4.1 ms), all within tolerance; with the long-term correlation of the whole envelopes speech is 81.1 ms, outside it. The speech example's spectral correlations are broader than the water example's (mean off-diagonal zero-lag correlation 0.464 against 0.259).**

## fig6-category-half-widths

> Water and wind sounds have fast temporal correlations (half widths 3.9 and 3.6 ms) and
> diagonal spectral correlations, while vocalizations have slow ones (speech 64.9 ms, dogs
> 76.3 ms, cats 117.3 ms) and more extensive spectral correlations.

In [ ]:
EXP6 = {"Flowing_Water": (3.9, 1), "Wind": (3.6, 1), "Human_Speech": (64.9, 6.49), "Dog_Barking": (76.3, 7.63), "Cat_Meowing": (117.3, 11.73)}
VOC = ["Single_Bird_Songs", "Cat_Meowing", "Dog_Barking", "Human_Speech"]
for lab, src in (("(ii)(a) 566 ms short-term", auto566), ("(ii)(b) long-term", autolong)):
    avg = {c: np.mean([src[(c, i)] for i in range(1, 16)], 0) for c in CATS + ["White_Noise"]}
    for j, wl in enumerate(("(iii)(a) mean curve", "(iii)(b) median of channels")):
        w = {c: widths(avg[c])[j] for c in avg}
        hits = sum(abs(w[c] - e) <= t for c, (e, t) in EXP6.items())
        pooled = widths(np.mean([avg[c] for c in ("Flowing_Water", "Wave", "Thunder_Rain")], 0))[j]
        sep = min(w[c] for c in VOC) > max(w["Flowing_Water"], w["Wind"])
        print(f"{lab}, {wl}: " + ", ".join(f"{c} {w[c]:.1f}" for c in EXP6) + f" | in tol {hits}/5 | every vocalization wider than water and wind: {sep}"
              f" | pooled water, waves, rain {pooled:.1f} | all: " + ", ".join(f"{c} {w[c]:.1f}" for c in avg if c not in EXP6))
cat_breadth = {c: np.mean([breadth((c, i))[0] for i in range(1, 16)]) for c in CATS + ["White_Noise"]}
print("mean off-diagonal zero-lag correlation by category (100 ms):", {c: round(float(v), 3) for c, v in sorted(cat_breadth.items(), key=lambda kv: kv[1])})

**Result: with the 566 ms short-term reading and the channel-averaged curve, four of five category half widths are within tolerance (flowing water 3.9, wind 3.9, speech 67.9, dogs 82.4 ms); cats are 160.9 ms against 117.3 +/- 11.7 ms. Every vocalization is wider than water and wind under every reading. Spectral breadth is highest for dogs, bells and speech (0.484, 0.436, 0.429) and lowest for flowing water (0.265), with single bird songs (0.273) low as well.**

## fig6c-category-diversity

> White noise has the lowest category diversity index; among natural sounds speech is lowest
> (CDI 0.12), crowd noise 0.15, wind 0.18, dogs 0.21, bird songs 0.29 and fire highest
> (0.39), with no clear trend between classes.

Time-averaged spectro-temporal correlation per sound at tau_W = 100 ms, 99 lags, 58 x 58
channels; readings (a) E||c_n - E c||^2 / ||E c||^2, (b) E||c_n - E c||^2 / E||c_n||^2, (c)
the same with norms not squared.

In [ ]:
EXPC = {"Human_Speech": 0.12, "Speech_Babble": 0.15, "Wind": 0.18, "Dog_Barking": 0.21, "Single_Bird_Songs": 0.29, "Fire": 0.39}
cdi = {r: {} for r in ("a", "b", "a_c", "b_c")}
for c in CATS + ["White_Noise"]:
    V = np.array([cbar[(c, i)].ravel().astype(np.float64) for i in range(1, 16)])
    m = V.mean(0)
    d2 = ((V - m) ** 2).sum(1)
    n2 = (V ** 2).sum(1)
    cdi["a"][c] = d2.mean() / (m ** 2).sum()
    cdi["b"][c] = d2.mean() / n2.mean()
    cdi["a_c"][c] = np.sqrt(d2).mean() / np.sqrt((m ** 2).sum())
    cdi["b_c"][c] = np.sqrt(d2).mean() / np.sqrt(n2).mean()
for r, v in cdi.items():
    hits = sum(abs(v[c] - e) <= 0.1 * e + 1e-9 for c, e in EXPC.items())
    nat = {c: x for c, x in v.items() if c != "White_Noise"}
    order = [c for c in sorted(EXPC, key=lambda c: v[c])]
    print(f"({r}): " + ", ".join(f"{c} {v[c]:.3f}" for c in EXPC) + f" | in tol {hits}/6 | lowest natural {min(nat, key=nat.get)},"
          f" white noise {v['White_Noise']:.3f} (rank {sorted(v.values()).index(v['White_Noise']) + 1} of 14) | rank order of the six kept:"
          f" {order == list(EXPC)}")
    print("      all:", {c: round(float(x), 3) for c, x in sorted(v.items(), key=lambda kv: kv[1])})

**Result: with norms not squared (reading (c)), five of the six CDIs are within tolerance (speech 0.123, crowd noise 0.144, wind 0.171, dogs 0.222, fire 0.381; single bird songs 0.373 against 0.29 +/- 0.029), speech is the lowest of the 13 categories, white noise is lower still (0.042), and the six keep the paper's rank order. With squared norms (readings (a) and (b)) the values are about ten times smaller (speech 0.016) and none is within tolerance.**

## fig7i-stationarity

> Environmental sounds other than fire (running water, waves, thunder, wind) are the most
> stationary natural sounds (SI = 0.49 +/- 0.04), vocalizations less so (0.28 +/- 0.04),
> speech is the least stationary category and white noise the most.

SI per sound from the short-term correlation at tau_W = 100 ms, 99 lags: (a) 1 -
<||c(t) - <c>||^2> / ||<c>||^2, (b) 1 - <||c(t) - <c>||^2> / <||c(t)||^2>.

In [ ]:
SI = {"a": {}, "b": {}}
for k, (e2, m2, n) in si_terms.items():
    var = e2 - m2
    SI["a"][k] = 1 - var / m2
    SI["b"][k] = 1 - var / e2
ENV_C = ["Flowing_Water", "Wave", "Thunder_Rain", "Wind"]
GROUPS = {"c": (ENV_C, VOC), "d": (ENV_C + ["Bird_Chorus", "Speech_Babble"], VOC)}
for r in ("a", "b"):
    catm = {c: np.mean([SI[r][(c, i)] for i in range(1, 16)]) for c in CATS + ["White_Noise"]}
    print(f"({r}) category means: " + ", ".join(f"{c} {v:.3f}" for c, v in sorted(catm.items(), key=lambda kv: kv[1])))
    for g, (env, voc) in GROUPS.items():
        for unit in ("e", "f"):
            if unit == "e":
                ev, vv = [catm[c] for c in env], [catm[c] for c in voc]
            else:
                ev = [SI[r][(c, i)] for c in env for i in range(1, 16)]
                vv = [SI[r][(c, i)] for c in voc for i in range(1, 16)]
            se = lambda x: np.std(x, ddof=1) / np.sqrt(len(x))
            ok = abs(np.mean(ev) - 0.49) <= 0.04 and abs(np.mean(vv) - 0.28) <= 0.04
            nat = {c: v for c, v in catm.items() if c != "White_Noise"}
            print(f"   ({r})({g})({unit}): environmental {np.mean(ev):.3f} +/- {se(ev):.3f}, vocalizations {np.mean(vv):.3f} +/- {se(vv):.3f}"
                  f" | both in tol {ok} | lowest category {min(nat, key=nat.get)}, highest overall {max(catm, key=catm.get)}")
    print(f"   ({r}) example sounds: speech 1 {SI[r][('Human_Speech', 1)]:.3f}, water 1 {SI[r][('Flowing_Water', 1)]:.3f}")

**Result: speech has the lowest category mean SI and white noise the highest under both readings, and environmental sounds are more stationary than vocalizations, but the values are not those reported: reading (a) gives 0.719 +/- 0.048 for the environmental categories and 0.358 +/- 0.089 for vocalizations (reported 0.49 +/- 0.04 and 0.28 +/- 0.04), reading (b) 0.792 and 0.621, and adding bird chorus and speech babble gives 0.654 under (a).**

## Model classification claims: cost estimate

Claims fig9a-optimal-resolution, fig9b-similar-maximum, fig9b-accumulation-rate,
fig9c-time-averaged-worse, fig9e-category-specific, fig9-spectrum-classifier-lower and
fig10-vocalization-background need the leave-one-sound-out GMM classifier at ten window
resolutions (25 to 566 ms) for three correlation classifiers and the spectrum classifier.
This cell times the parts on a subset and estimates the whole, under the cheapest readings:
the reported PC and component counts used at every resolution (reading (7)(b), no search over
N_c = 1 to 20), and training observations every tau_W / 4 or, cheaper still, at
non-overlapping windows.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA as skPCA
RES = 25 * 2 ** (np.arange(10) / 2)
fs, x, _ = sounds[("Human_Speech", 1)]
Sg = M.cochleogram(x, fs)
t_feat = []
for tw in RES:
    t1 = time.time()
    z = np.array(list(M.short_term_iter(Sg, tw, 0, pairs="zero")))
    a = np.array(list(M.short_term_iter(Sg, tw, int(tw / 2), pairs="auto")))
    t_feat.append(time.time() - t1)
coch_t = []
for key in [("Fire", 1), ("Wind", 2)]:
    t1 = time.time(); M.cochleogram(sounds[key][1], 44100); coch_t.append(time.time() - t1)
feat_total = 195 * (np.sum(t_feat) + np.mean(coch_t))
print("feature time for one sound over the ten resolutions (s):", np.round(t_feat, 2), "; cochleogram", np.round(np.mean(coch_t), 2))
print(f"features for 195 sounds at ten resolutions: {feat_total / 60:.1f} min")
iu = np.triu_indices(58, 1)
rng_t = np.random.default_rng(7)
fit_t = {}
for step_lab, per_sound in (("tau_W/4 at 141 ms", int(10000 / (141 / 4))), ("non-overlapping 141 ms", int(10000 / 141))):
    Xc = rng_t.standard_normal((14 * per_sound, 26))
    t1 = time.time()
    GaussianMixture(8, covariance_type="diag", random_state=0, max_iter=100).fit(Xc)
    fit_t[step_lab] = time.time() - t1
nfits = 195 * 13 * 3 * 10
for k, v in fit_t.items():
    print(f"one GMM fit (8 diagonal components, 26 PCs, {k} training observations of one category): {v:.2f} s;"
          f" {nfits} fits (195 folds x 13 categories x 3 classifiers x 10 resolutions): {nfits * v / 60:.0f} min")
# reading (7)(a): N_c chosen by cross-validated likelihood over 1 to 20 components, at least one
# 5-fold search per category, classifier and resolution, with fits whose cost grows with N_c
n_search = 13 * 3 * 10 * 20 * 5
for k, v in fit_t.items():
    print(f"reading (7)(a) adds at least {n_search} fits ({k}): {n_search * v * 10.5 / 8 / 60:.0f} min")
elapsed = (time.time() - T_START) / 60
v_lo, v_hi = min(fit_t.values()), max(fit_t.values())
lo = elapsed + feat_total / 60 + (nfits + n_search * 10.5 / 8) * v_lo / 60
hi = elapsed + feat_total / 60 + (nfits + n_search * 10.5 / 8) * v_hi / 60
print(f"elapsed so far in this notebook: {elapsed:.1f} min; with the classifier the notebook would take about {lo:.0f} to {hi:.0f} min")

**Result: the leave-one-sound-out classifier needs features at ten resolutions and at least 76050 GMM fits even with the reported PC and component counts, and choosing N_c by cross-validation, as the Methods say, adds at least 39000 more. The cell prints the estimated duration of this notebook with the classifier, which changes from run to run with the machine (between about one and four hours over the runs made); under the more expensive readings it is over the 60 minute limit, and building and running the classifier does not fit the time left in this run. The seven classification claims are blocked as too large.**

## Summary

| Claim | Outcome |
|---|---|
| fig5-speech-water-widths | reproduced |
| fig6-category-half-widths | reproduced |
| fig6c-category-diversity | reproduced (norms not squared) |
| fig7i-stationarity | not reproduced (order holds, values do not) |
| fig9a, fig9b (two), fig9c, fig9e, fig9 spectrum, fig10 | blocked, too large for this run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the Methods' critical bandwidths are used where the
code (criticalbandwidth.m) has another formula; envelopes are at 1 kHz as the Methods say (the
code's example keeps 3675 Hz); the short-term correlation is sampled every tau_W / 4; white noise
is generated. What was found about the dataset: the 195 sounds, 15 per category, 10 s at
44.1 kHz, are all there, as 16 bit integer WAV files where the description says double
precision; white noise is not included.

In [ ]:
print(f"total run time {(time.time() - T_START) / 60:.1f} min")